# Task 1 — Notebook 01: Data Ingestion (CDL + NDVI)

**Purpose (NAFSI Task 1):** Load **CDL** as a crop mask and **MODIS NDVI** time series for the Corn Belt (Iowa / Nebraska focus in config), compare corn vs. soybean phenology. Data are read from **processed wide Parquet** produced by `scripts/process_interim_to_parquet.py` (EPSG:5070).

**Inputs (processed):**  
- `data/processed/cdl/cdl_stack_wide.parquet` — one row per pixel (`iy`, `ix`), one column per CDL year  
- `data/processed/ndvi/ndvi_weekly_{year}_wide.parquet` — one row per pixel, weekly columns (`w000`, `w001`, …)
- JSON sidecars with CRS, transform, and time-column mappings

**This notebook:** loads CDL + NDVI for the analysis year from `configs/task1_ndvi_analysis.yaml`, converts wide Parquet to `xarray`. QA, smoothing, and exports are in later notebooks.

In [4]:
# NOTE: This notebook was scaffolded with AI assistance. All analysis logic
# and interpretation must be reviewed and authored by the team.

import json
import sys
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
import xarray as xr
import yaml
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 48)

_cwd = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_cwd, *_cwd.parents) if (p / "requirements.txt").is_file() and (p / "src").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not find repo root (requirements.txt + src/). cd to the repo and retry.")
sys.path.insert(0, str(REPO_ROOT))

with open(REPO_ROOT / "configs" / "task1_ndvi_analysis.yaml") as f:
    cfg = yaml.safe_load(f)

cdl_year = cfg["cdl"]["year"]
ndvi_year = int(cfg["ndvi"]["year"])
print("Config:", "CDL year", cdl_year, "| NDVI year", ndvi_year)

Config: CDL year 2022 | NDVI year 2022


In [5]:
# CDL from processed Parquet — wide table with one column per year (cdl_YYYY).
cdl_parquet_path = REPO_ROOT / "data" / "processed" / "cdl" / "cdl_stack_wide.parquet"
if not cdl_parquet_path.exists():
    raise FileNotFoundError(
        f"CDL parquet not found at {cdl_parquet_path}. "
        "Run: python scripts/process_interim_to_parquet.py --dataset cdl"
    )

cdl_df = pd.read_parquet(cdl_parquet_path)
coord_cols_cdl = [c for c in ["y", "x", "lat", "lon", "iy", "ix"] if c in cdl_df.columns]
cdl_ds = cdl_df.set_index(coord_cols_cdl).to_xarray() if coord_cols_cdl else cdl_df.to_xarray()

cdl_var = f"cdl_{cdl_year}"
if cdl_var not in cdl_ds:
    raise ValueError(f"Config CDL year column '{cdl_var}' not in parquet. Available: {list(cdl_ds.data_vars)}")
cdl = cdl_ds[cdl_var]
print("cdl_ds (all years)", dict(cdl_ds.sizes), "| cdl (mask year)", cdl_var, dict(cdl.sizes))

cdl_ds (all years) {'iy': 1520, 'ix': 2048} | cdl (mask year) cdl_2022 {'iy': 1520, 'ix': 2048}


In [6]:
# NDVI from processed Parquet — wide table with weekly columns (w000, w001, …).
ndvi_parquet_path = REPO_ROOT / "data" / "processed" / "ndvi" / f"ndvi_weekly_{ndvi_year}_wide.parquet"
if not ndvi_parquet_path.exists():
    raise FileNotFoundError(
        f"NDVI parquet not found at {ndvi_parquet_path}. "
        f"Run: python scripts/process_interim_to_parquet.py --dataset ndvi --year {ndvi_year}"
    )

ndvi_df = pd.read_parquet(ndvi_parquet_path)
coord_cols_ndvi = [c for c in ["y", "x", "lat", "lon", "iy", "ix"] if c in ndvi_df.columns]
time_cols_ndvi = [c for c in ndvi_df.columns if c not in coord_cols_ndvi]
ndvi_df = ndvi_df.melt(
    id_vars=coord_cols_ndvi, value_vars=time_cols_ndvi,
    var_name="time", value_name="ndvi",
)

if coord_cols_ndvi:
    ndvi = ndvi_df.set_index(["time"] + coord_cols_ndvi).to_xarray()
else:
    ndvi = ndvi_df.to_xarray()

print("ndvi (analysis year)", dict(ndvi.sizes), "| file:", ndvi_parquet_path.name)

ndvi (analysis year) {'time': 23, 'iy': 1520, 'ix': 2048} | file: ndvi_weekly_2022_wide.parquet


In [7]:
# Parquet drops CRS; re-apply the pipeline's EPSG:5070 (CONUS Albers).
try:
    import rioxarray  # noqa: F401

    cdl_ds = cdl_ds.rio.write_crs("EPSG:5070")
    ndvi = ndvi.rio.write_crs("EPSG:5070")

    print("CDL CRS:", getattr(cdl_ds.rio, "crs", None))
    print("NDVI CRS:", getattr(ndvi.rio, "crs", None))
except Exception as exc:
    print("CRS check skipped (optional rioxarray .rio):", exc)

CDL CRS: EPSG:5070
NDVI CRS: EPSG:5070


## Processed Parquet → xarray

`cdl_ds` / `ndvi` are **xarray** objects built from the wide Parquet files. Below: dimension summary, Parquet schema (every column name + type), JSON sidecar metadata, and a short sample table.

In [8]:
def _ds_profile(name: str, obj) -> pd.DataFrame:
    sizes = dict(obj.sizes)
    if isinstance(obj, xr.Dataset):
        dvars = ", ".join(str(v) for v in obj.data_vars)
    else:
        dvars = str(obj.name) if obj.name else "—"
    return pd.DataFrame(
        [
            {"object": name, "field": "dims (sizes)", "value": str(sizes)},
            {"object": name, "field": "data_vars / name", "value": dvars},
            {"object": name, "field": "coords", "value": ", ".join(map(str, obj.coords))},
        ]
    )


def _parquet_columns(path: Path) -> pd.DataFrame:
    pf = pq.ParquetFile(path)
    sch = pf.schema_arrow
    rows = [{"column": name, "pyarrow_type": str(sch.field(name).type)} for name in sch.names]
    out = pd.DataFrame(rows)
    out.insert(0, "parquet_file", path.name)
    return out


def _parquet_head(path: Path, n: int = 8) -> pd.DataFrame:
    pf = pq.ParquetFile(path)
    batch = next(pf.iter_batches(batch_size=n))
    return batch.to_pandas()


print("— xarray objects (from processed Parquet) —")
display(pd.concat([_ds_profile("cdl_ds (all years)", cdl_ds), _ds_profile("cdl (mask year)", cdl)], ignore_index=True))
display(_ds_profile("ndvi (analysis year)", ndvi))

cdl_meta = REPO_ROOT / "data" / "processed" / "cdl" / "cdl_stack_spatial_metadata.json"
ndvi_meta = REPO_ROOT / "data" / "processed" / "ndvi" / f"ndvi_weekly_{ndvi_year}_metadata.json"

print("\n— CDL processed Parquet (all columns) —")
pf = pq.ParquetFile(cdl_parquet_path)
print(f"rows (approx): {pf.metadata.num_rows:,} | row_groups: {pf.num_row_groups}")
display(_parquet_columns(cdl_parquet_path))
display(_parquet_head(cdl_parquet_path, n=8))

if cdl_meta.is_file():
    print("\nCDL spatial metadata JSON (flattened)")
    display(pd.json_normalize(json.loads(cdl_meta.read_text(encoding="utf-8")), sep="_"))

print(f"\n— NDVI processed Parquet ({ndvi_year}) — all columns —")
pf = pq.ParquetFile(ndvi_parquet_path)
print(f"rows (approx): {pf.metadata.num_rows:,} | row_groups: {pf.num_row_groups}")
display(_parquet_columns(ndvi_parquet_path))
display(_parquet_head(ndvi_parquet_path, n=8))

if ndvi_meta.is_file():
    print("\nNDVI metadata JSON (flattened)")
    meta = json.loads(ndvi_meta.read_text(encoding="utf-8"))
    flat = {k: v for k, v in meta.items() if k != "time_start_day"}
    display(pd.json_normalize([flat], sep="_"))
    tsd = meta.get("time_start_day")
    if isinstance(tsd, list) and len(tsd) > 0:
        tdf = pd.DataFrame({"week_index": range(len(tsd)), "time_start_day": tsd})
        print("Per-week column mapping: w000 … ↔ time_start_day (head / tail)")
        display(pd.concat([tdf.head(6), tdf.tail(6)], ignore_index=True))

— xarray objects (from processed Parquet) —


,object,field,value
0,cdl_ds (all years),dims (sizes),"{'iy': 1520, 'ix': 2048}"
1,cdl_ds (all years),data_vars / name,"cdl_2008, cdl_2009, cdl_2010, cdl_2011, cdl_..."
2,cdl_ds (all years),coords,"iy, ix, spatial_ref"
3,cdl (mask year),dims (sizes),"{'iy': 1520, 'ix': 2048}"
4,cdl (mask year),data_vars / name,cdl_2022
5,cdl (mask year),coords,"iy, ix"


,object,field,value
0,ndvi (analysis year),dims (sizes),"{'time': 23, 'iy': 1520, 'ix': 2048}"
1,ndvi (analysis year),data_vars / name,ndvi
2,ndvi (analysis year),coords,"time, iy, ix, spatial_ref"



— CDL processed Parquet (all columns) —
rows (approx): 3,112,960 | row_groups: 3


,parquet_file,column,pyarrow_type
0,cdl_stack_wide.parquet,iy,int32
1,cdl_stack_wide.parquet,ix,int32
2,cdl_stack_wide.parquet,cdl_2008,int32
3,cdl_stack_wide.parquet,cdl_2009,int32
4,cdl_stack_wide.parquet,cdl_2010,int32
5,cdl_stack_wide.parquet,cdl_2011,int32
6,cdl_stack_wide.parquet,cdl_2012,int32
7,cdl_stack_wide.parquet,cdl_2013,int32
8,cdl_stack_wide.parquet,cdl_2014,int32
9,cdl_stack_wide.parquet,cdl_2015,int32


...

,iy,ix,w000,w001,w002,w003,w004,w005,w006,w007,w008,w009,w010,w011,w012,w013,w014,w015,w016,w017,w018,w019,w020,w021,w022
0,0,0,178.0,176.0,175.0,175.0,183.0,184.0,212.0,180.0,175.0,184.0,159.0,159.0,174.0,160.0,163.0,158.0,155.0,157.0,162.0,160.0,166.0,175.0,178.0
1,0,1,178.0,171.0,178.0,173.0,183.0,179.0,212.0,180.0,170.0,174.0,159.0,157.0,173.0,160.0,160.0,156.0,154.0,156.0,162.0,157.0,161.0,175.0,178.0
2,0,2,175.0,171.0,170.0,170.0,178.0,174.0,208.0,179.0,170.0,169.0,161.0,157.0,166.0,158.0,160.0,156.0,155.0,153.0,162.0,155.0,158.0,163.0,165.0
3,0,3,173.0,172.0,171.0,171.0,181.0,176.0,211.0,174.0,169.0,166.0,162.0,156.0,167.0,158.0,161.0,153.0,153.0,154.0,159.0,156.0,160.0,161.0,222.0
4,0,4,172.0,173.0,169.0,170.0,178.0,173.0,207.0,174.0,167.0,166.0,160.0,159.0,166.0,157.0,157.0,154.0,153.0,154.0,159.0,156.0,162.0,164.0,194.0
5,0,5,171.0,173.0,169.0,172.0,178.0,178.0,207.0,177.0,170.0,166.0,160.0,159.0,166.0,156.0,157.0,155.0,154.0,154.0,160.0,160.0,162.0,164.0,209.0
6,0,6,170.0,175.0,172.0,173.0,182.0,182.0,212.0,183.0,175.0,170.0,165.0,163.0,165.0,157.0,157.0,155.0,154.0,154.0,159.0,159.0,160.0,166.0,171.0
7,0,7,171.0,176.0,174.0,174.0,193.0,181.0,214.0,182.0,175.0,172.0,168.0,163.0,164.0,156.0,157.0,155.0,154.0,154.0,159.0,157.0,160.0,166.0,172.0



NDVI metadata JSON (flattened)


,source_nc,year,height,width,n_time,wide_columns,note,crs,transform
0,data\interim\ndvi\ndvi_weekly_2022.nc,2022,1520,2048,23,"[iy, ix, w000, w001, w002, w003, w004, w005,...",w000.. are weekly NDVI layers in the same or...,None,"[320.3125, 0.0, -843000.0, 0.0, -319.0789473..."


Per-week column mapping: w000 … ↔ time_start_day (head / tail)


,week_index,time_start_day
0,0,2022-05-02
1,1,2022-05-09
2,2,2022-05-16
3,3,2022-05-23
4,4,2022-05-30
5,5,2022-06-06
6,17,2022-08-29
7,18,2022-09-05
8,19,2022-09-12
9,20,2022-09-19
